# `playlistsmith`


If you are like me, you probably have a playlist on your favourite streaming platform that just contains any song you listened to and liked at some point. Now, imagine further that one day you feel like listening to a specific type of song—the one that exactly fits your mood in this moment. However, if you have had the "just-dump-songs-into-one-playlist" habit for some time, your playlist is just a mess. It contains all types of songs, and it would be tedious to find one that fits your current mood, let alone sort this playlist by hand so you will not have this problem next time. But don't worry, there might be a way out of this cycle, and it's called `playlistsmith`.

`playlistsmith` is a Python package with a graphical user interface (GUI) that allows you to find sub-playlists in your playlist in just a few steps. These steps are discussed in turn below. I also added a running example, so you can see the package/GUI in action. Note that I will walk through the GUI first and then briefly discuss how to obtain similar results using the package directly in Python. If you need more detailed information for any of the steps discussed below, please consult the package documentation.

**DISCLAIMER**: This package has been developed to work with Spotify playlists. However, an implementation for other platforms is planned in the future.

## 0. Requirements
Before you can get started with using `playlistsmith`, you need to have the following set up:

1. Python 3.12
2. Internet access
3. If 1 and 2 apply, you can install `playlistsmith` from source including access to the gui by running:

```bash
python3.12 -m pip install "playlistsmith[gui] @ git+https://github.com/Programming-The-Next-Step-2026/playlistsmith"
```

This will also install the following dependencies: numpy, pandas, scikit-learn, scipy, umap-learn, httpx, streamlit>=1.33, plotly>=5.18, matplotlib

## 1. Obtain a .csv of your playlist
The package expects a .csv of the format that [Exportify](https://exportify.app/) provides. You can log-in with your Spotify account to see a list of your playlists. Simply press the "Export" button on the right, and you are ready to go. 

## 2. Start up the GUI
From the environment that you installed `playlistsmith` into, run the following in a terminal:

```bash
playlistsmith-gui
```

Note: If you do not have internet access, there is also a demo mode that mocks API access. Simply run:

```bash
playlistsmith-gui --demo
```

The streamlit application should open up in your browser shortly.

## 3. Upload the .csv
After starting up the GUI, you'll find yourself looking at the screen below. Note that you can ignore the sidebar as only one feature extraction method is implemented at the moment.

<img src="./gui_screenshots/csv_upload.png" alt="csv_upload" width="800">

If you press the "Upload" button (indicated by the red box in the screenshot), you can select a .csv from your local drive to upload. Choose the Exportify generated file here.

### 3.1 Running example
For illustration, I created a mix of classical and rock music. After uploading the data, you can inspect the first 20 songs using the "Preview tracklist" expander. The first songs of the running example are shown below.

<img src="./gui_screenshots/tracklist_preview.png" alt="tracklist_preview" width="800">

## 4. Extract features
Scrolling down will reveal an expander with a list of features and an "Extract features" button. This will query the [ReccoBeats](https://reccobeats.com/) API for precomputed features.

<img src="./gui_screenshots/feature_extraction.png" alt="feature_extraction" width="800">

### 4.1 Running example
Extracting features from the example playlist induces the following coverage report:

<img src="./gui_screenshots/coverage_report.png" alt="coverage_report" width="800">

Songs included in the table are not covered by ReccoBeats and therefore excluded from the analysis.

## 5. Clustering
Now, this is where the magic happens: running a clustering algorithm on the playlist to discover sub-collections of songs. `playlistsmith` offers three algorithms: a Gaussian Mixture Model (GMM), K-means and Hierarchical Density-Based Spatial Clustering of Applications with Noise (HDBSCAN). While the algorithms are described in a bit more detail in the info strings accessible by hovering over the "?" icon, suffice it to say here that GMM is suitable for most applications, K-means can be run in small samples or as a sanity check in addition to the GMM, and HDBSCAN is useful if you suspect that your playlist includes songs that are unique so that they would likely not fit into any sub-playlist. 

<img src="./gui_screenshots/algorithms.png" alt="algorithms" width="800">

Before we can start clustering, we should take a minute to think about hyperparameters—settings that influence the outcome of each algorithm (more information accessible by hovering over the "?"). As we are going to run the GMM below, let's quickly discuss the choices made:

1. `min_playlist_size`: What is the minimum amount of songs you would like to have in the sub-playlists? Smaller clusters will be assigned to an unclassified class. We stick with 5 (the default).
2. `max_playlist_share`: The proportion of uploaded tracks a sub-playlist is maximally allowed to contain before returning a warning. As we know that two playlists underlie our running example and that they are unequal in size, the default (0.5) seems unreasonable. We opt for 0.65.
3. `k range`: The range of cluster numbers to test and compare. The algorithm will be fit to discover each number of clusters within this range and the best fit will be retained. As we have a clear idea about how many clusters we would like to find, we select 2 by just dragging the knobs on top of each other<sup id="ref1-c7"><a href="#fn1-c7">1</a></sup>.

<img src="./gui_screenshots/hyperparameters_cluster.png" alt="hyperparameters_cluster" width="800">

Press "Cluster" to run the algorithm.


---
<p id="fn1-c7" style="font-size:0.85em"><sup>1</sup> You can also try to fit a range of clusters, but the GMM will still discover that two clusters fit the data best. <a href="#ref1-c7">↩</a></p>


## 6. Diagnostics & visualisation
### 6.1 Running example
If you select a range for k, the number of found clusters is determined based on fit indices. The "Per-cluster z-profile heatmap" indicates how songs in each cluster score on the extracted features on average.

<img src="./gui_screenshots/heatmap.png" alt="heatmap" width="800">

In the present example, Cluster 1 (bottom row) seems to score high on energy and loudness while Cluster 0 scores low on both of these features. This provides some evidence that Cluster 1 contains the rock songs whereas Cluster 0 is made up of classical songs. This is supported by the visualisation output in the GUI. Hovering over the data points shows the song title as well as artist names.

<img src="./gui_screenshots/cluster_visualisation.png" alt="cluster_visualisation" width="800">

A quick scan through both clusters confirms that Cluster 0 consists of classical songs and Cluster 1 of rock songs.

## 7. Exporting playlists
The final stage in the GUI is exporting the created playlists as .csv files. 

### 7.1 Running example
<img src="./gui_screenshots/export.png" alt="export" width="800">

You can specify the output directory<sup id="ref1-c9"><a href="#fn1-c9">1</a></sup>, choose to also export the unclassified songs<sup id="ref2-c9"><a href="#fn2-c9">2</a></sup>, or write a combined file that includes the cluster assignment in a separate column. Additionally, you can rename the .csv files (last column in the table). Ready to export? Simply click "Write CSVs".


---
<p id="fn1-c9" style="font-size:0.85em"><sup>1</sup> If the directory does not exist, it will be created. <a href="#ref1-c9">↩</a></p>
<p id="fn2-c9" style="font-size:0.85em"><sup>2</sup> This is common when using HDBSCAN or when a playlist smaller than <code>min_playlist_size</code> is discovered. <a href="#ref2-c9">↩</a></p>


## 8. Importing the playlists into Spotify
Only one question remaining: How do I get the playlists back into my Spotify? This turns out to be rather simple.
1. Open the .csv you just exported with your favourite spreadsheet program.
2. Select and copy the column containing the Track URIs.
3. Open the Spotify desktop app and create a new playlist. 
4. Press CMD + V/CTRL + V to paste the songs into the playlist. 

And voilà, you have split up your tediously long playlist into smaller chunks. Hopefully, you'll find the right song for your mood faster next time! Want to split up another playlist? Press "Reset session" in the sidebar, and you can start fresh.

If you are only interested in using the GUI, you should be ready to go now. For users who prefer the pure Python experience and who are interested in the internal workings of `playlistsmith`, I'll quickly mirror the above-mentioned steps using the underlying package.

## Bridging GUI and package

Before turning to the Python usage, here is how the GUI steps you just saw map onto the underlying functions. The right column (*User*) is exactly the click-path from sections 1–8. The left column (*Software*) is the code each click triggers. The dashed box marks everything that a single `ps.cluster(...)` call wraps: Preprocessing, the fit, ordering, the small-cluster collapse and interpretation all happen inside that one function. The pure-Python usage in the next section walks the same software path, just through calling `playlistsmith`'s publicly exported functions.

<img src="./gui_screenshots/mermaid_diagram.png" alt="mermaid_diagram" width="1000">

## Using the Python package
## 1. Read the .csv

In [27]:
import playlistsmith as ps

tl = ps.TrackLibrary('rock_&_classical_mix.csv')
tl.display() # show sample of songs in the library

                                                    title                                                                                         artist              spotify_id
0                                                The Kill                                                                         Thirty Seconds To Mars  4rRNDclay9ayn1iR1VpMMB
1                                                    Numb                                                                                    Linkin Park  2nLtzopw4rPReszdYBJU6h
2                              Boulevard of Broken Dreams                                                                                      Green Day  5GorCbAP4aL0EJ16frG2hd
3                                              Can't Stop                                                                          Red Hot Chili Peppers  3ZOEytgrvLwQaqXreDs2Jx
4                                        Bring Me To Life                                                          

## 2. Extract features

In [28]:
features, coverage = tl.extract_features(mode = "precomputed") # only precomputed is implemented right now

[playlistsmith] Dropped 34 track(s) with no ReccoBeats precomputed features:
  - The Kill — Thirty Seconds To Mars
  - Boulevard of Broken Dreams — Green Day
  - It's Been Awhile — Staind
  - I Write Sins Not Tragedies — Panic! At The Disco
  - Decode - Twilight Soundtrack Version — Paramore
  - Joker And The Thief — Wolfmother
  - No One Knows — Queens of the Stone Age
  - Obstacle 1 — Interpol
  - Woman — Wolfmother
  - Tick Tick Boom — The Hives
  - Gerropaedie — Sally Beamish, Stephanie Irvine
  - Innocence - Nowhere Sessions — Snorri Hallgrímsson
  - Touch Her Soft Lips and Part (From "Henry V") — William Walton, John Rutter, Aurora Orchestra
  - Positano — Stephan Moccio
  - No. 2, Kum Ba Ya — Adolphus Hailstork, Virginia Symphony Orchestra, Joann Falletta
  - Prélude, M. 65 — Maurice Ravel, Mao Fujita
  - Lento, ma non troppo — Frederick Delius, Northern Sinfonia, David Lloyd-Jones
  - Patterns / Solo - Pt. 2 / Faded — Max Richter, Louisa Fuller, Natalia Bonner, Nick Barr, Max R

In [29]:
print(features.columns) 
print(coverage) # short summary
print(coverage.dropped_tracks.head(5)) # full list of dropped tracks

Index(['spotify_id', 'title', 'artist', 'acousticness', 'danceability',
       'energy', 'instrumentalness', 'liveness', 'loudness', 'speechiness',
       'tempo', 'valence'],
      dtype='str')
Feature coverage: 205/239 track(s) resolved via ReccoBeats; 34 dropped.
               spotify_id                                 title  \
0  4rRNDclay9ayn1iR1VpMMB                              The Kill   
1  5GorCbAP4aL0EJ16frG2hd            Boulevard of Broken Dreams   
2  25CMmGsl22APKhfuj4Tp7j                      It's Been Awhile   
3  4bPQs0PHn4xbipzdPfn6du            I Write Sins Not Tragedies   
4  4IDfVjI1TlB1UwlC01T4Bm  Decode - Twilight Soundtrack Version   

                   artist  
0  Thirty Seconds To Mars  
1               Green Day  
2                  Staind  
3     Panic! At The Disco  
4                Paramore  


## 3. Clustering

In [30]:
clust = ps.cluster(
    features_df = features,
    method = "gmm", # or "kmeans", "hdbscan"
    min_playlist_size = 5,
    max_playlist_share = 0.65,
    k_range = range(2, 3), # only 2 clusters for this example, but you can try more
)

print(clust.warnings) # no warnings for this example
print(clust.tracks.head(5)) # tracks with cluster assignments and cluster summary
print(clust.descriptions) # cluster infos

[]
               spotify_id             title                 artist  cluster  \
0  2nLtzopw4rPReszdYBJU6h              Numb            Linkin Park        1   
1  3ZOEytgrvLwQaqXreDs2Jx        Can't Stop  Red Hot Chili Peppers        1   
2  0COqiPhxzoWICwFCS4eZcp  Bring Me To Life            Evanescence        1   
3  4VqPOruhp5EdPBeR92t6lQ          Uprising                   Muse        1   
4  6GG73Jik4jUlQCkKg9JuGO        The Middle        Jimmy Eat World        1   

                                   cluster_summary  
0  low acousticness, high danceability, fast tempo  
1  low acousticness, high danceability, fast tempo  
2  low acousticness, high danceability, fast tempo  
3  low acousticness, high danceability, fast tempo  
4  low acousticness, high danceability, fast tempo  
   cluster  size                         top_features  \
0        0   115         [acousticness, danceability]   
1        1    90  [acousticness, danceability, tempo]   

                                

The clustering returns two major classes: `ClusterDiagnostics` and `ClusteringResult`.

In [31]:
clust_diag = clust.diagnostics
print(clust_diag.zprofile_heatmap) # z-profile that also shown in the GUI
print(clust_diag.projection_2d) # 2D projection of the feature space

   acousticness  danceability    energy  instrumentalness  liveness  loudness  \
0      0.847826     -0.558034 -0.825633          0.813648 -0.426072 -0.819425   
1     -1.083333      0.713043  1.054976         -1.039662  0.544425  1.047043   

   speechiness     tempo   valence  
0    -0.285088 -0.432451 -0.671764  
1     0.364279  0.552577  0.858365  
         dim1      dim2
0    7.941162  7.474200
1    6.661907  6.717361
2    6.609728  8.128035
3    5.707242  8.011624
4    5.086491  6.726296
..        ...       ...
200 -6.273704  5.451881
201 -6.500846  8.053660
202 -3.756524  5.815954
203 -6.139884  5.351421
204 -6.460508  6.878891

[205 rows x 2 columns]


In [32]:
clust_res = clust.clustering # contains results and fit indices
print(clust_res.labels) # cluster assignment for each track
print(clust_res.bic_curve) # BIC value(s)

[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
{2: 3074.862211084461}


## 4. Write .csvs

In [33]:
import pandas as pd

ps.io.write_cluster_csvs(
    result = clust,
    output_dir = "../playlistsmith_out",
    naming = {0: "classical", 1: "rock"},
)

# Inspect the generated CSV files
print(pd.read_csv("../playlistsmith_out/classical.csv").head(2))
print(pd.read_csv("../playlistsmith_out/rock.csv").head(2))

                              Track URI  \
0  spotify:track:1cmigB9I6IRpFqjIbzvSQB   
1  spotify:track:3U8Fx7zNTQrctytkj6Gqgd   

                                          Track Name  \
0       Suite bergamasque, L. 75: III. Clair de lune   
1  The Carnival of the Animals, R. 125: XIII. The...   

                                      Artist Name(s)  Cluster  \
0                     Claude Debussy, Alice Sara Ott        0   
1  Camille Saint-Saëns, Isata Kanneh-Mason, Jeneb...        0   

                       Cluster Summary  
0  high acousticness, low danceability  
1  high acousticness, low danceability  
                              Track URI  Track Name         Artist Name(s)  \
0  spotify:track:2nLtzopw4rPReszdYBJU6h        Numb            Linkin Park   
1  spotify:track:3ZOEytgrvLwQaqXreDs2Jx  Can't Stop  Red Hot Chili Peppers   

   Cluster                                  Cluster Summary  
0        1  low acousticness, high danceability, fast tempo  
1        1  low acousti